# IMDB Sentiment Analysis using Bidirectional LSTM

This project performs **binary sentiment classification** on the [IMDB movie review dataset](https://huggingface.co/datasets/stanfordnlp/imdb) using a **Bidirectional Long Short-Term Memory (BiLSTM)** neural network built with PyTorch.

### Project pipeline

**IMDB reviews → text preprocessing → vocabulary → token IDs → padded sequences → Embedding → BiLSTM → fully connected layers → sentiment prediction**

### Final experiment

- Training data: 20,000 reviews
- Validation data: 5,000 reviews
- Test data: 25,000 reviews
- Maximum sequence length: 300 tokens
- Batch size: 64
- Embedding dimension: 20
- BiLSTM hidden size: 64 per direction
- Fully connected hidden size: 64
- Optimizer: Adam
- Learning rate: 0.002
- Epochs: 10
- Training hardware: NVIDIA Tesla T4 via Google Colab


## 1. Environment and Hardware

The model was trained in Google Colab using an NVIDIA GPU. The notebook automatically falls back to CPU if a GPU is unavailable.


In [28]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU available. Training will use the CPU.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU count: 1


## 2. Install Dependencies

The project uses Hugging Face `datasets` to load the IMDB dataset. `torchtext` is intentionally not used because it is deprecated and is unnecessary for this implementation.


In [ ]:
!pip install -q datasets


## 3. Load the IMDB Dataset

The standard IMDB dataset contains 25,000 labeled training reviews and 25,000 labeled test reviews. The training set is later split into 20,000 training samples and 5,000 validation samples.


In [31]:
from datasets import load_dataset

train_dataset, test_dataset = load_dataset(
    "stanfordnlp/imdb",
    split=["train", "test"]
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))


Training samples: 25000
Test samples: 25000


## 4. Text Preprocessing and Tokenization

The tokenizer:

1. Removes HTML tags such as `<br />`.
2. Extracts common emoticons.
3. Converts text to lowercase.
4. Removes non-word characters.
5. Splits the cleaned text into tokens.


In [ ]:
import re

def tokenizer(text):
    """Clean and tokenize an IMDB review."""
    text = re.sub(r"<[^>]*>", "", text)

    emoticons = re.findall(
        r"(?::|;|=)(?:-)?(?:\)|\(|D|P)",
        text.lower()
    )

    text = re.sub(r"[\W]+", " ", text.lower())
    text += " " + " ".join(emoticons).replace("-", "")

    return text.split()

# Example tokenization
sample = train_dataset[0]["text"]
print(tokenizer(sample)[:30])


## 5. Build the Vocabulary

A vocabulary is created from the original **25,000-review IMDB training partition** before the 20,000/5,000 train-validation split. The labels are not used when building the vocabulary. Two special tokens are reserved:

- `<pad>` → `0`, used for sequence padding
- `<unk>` → `1`, used for tokens not present in the vocabulary


In [ ]:
from collections import Counter

counter = Counter()

for example in train_dataset:
    counter.update(tokenizer(example["text"]))

print("Unique tokens:", len(counter))

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token in counter:
    vocab[token] = len(vocab)

UNK_IDX = vocab[UNK_TOKEN]
PAD_IDX = vocab[PAD_TOKEN]

print("Vocabulary size:", len(vocab))


## 6. Encode Reviews and Create Train/Validation Split

Each review is converted from text into integer token IDs. The original 25,000-review training set is split into 20,000 training samples and 5,000 validation samples using a fixed split seed.


In [ ]:
def encode_review(text):
    tokens = tokenizer(text)
    return [vocab.get(token, UNK_IDX) for token in tokens]

from torch.utils.data import random_split

train_subset, valid_subset = random_split(
    train_dataset,
    [20_000, 5_000],
    generator=torch.Generator().manual_seed(42)
)

print("Training:", len(train_subset))
print("Validation:", len(valid_subset))
print("Test:", len(test_dataset))


## 7. Batch Preparation

Reviews have different lengths, so each batch is padded to the longest review in that batch. To keep LSTM computation manageable, reviews are truncated to a maximum of **300 tokens**.


In [ ]:
MAX_LENGTH = 300

def collate_batch(batch):
    labels = []
    encoded_reviews = []

    for example in batch:
        review = encode_review(example["text"])[:MAX_LENGTH]

        if len(review) == 0:
            review = [UNK_IDX]

        encoded_reviews.append(review)
        labels.append(example["label"])

    lengths = torch.tensor(
        [len(review) for review in encoded_reviews],
        dtype=torch.long
    )

    max_len = max(lengths).item()

    padded = torch.full(
        (len(encoded_reviews), max_len),
        PAD_IDX,
        dtype=torch.long
    )

    for i, review in enumerate(encoded_reviews):
        padded[i, :len(review)] = torch.tensor(review)

    labels = torch.tensor(labels, dtype=torch.float32)

    return padded, lengths, labels


## 8. DataLoaders


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

valid_loader = DataLoader(
    valid_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)


## 9. Select the Training Device


In [14]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Using device: cuda
GPU: Tesla T4


## 10. Define the Bidirectional LSTM Model

The model uses an embedding layer followed by a bidirectional LSTM. The final hidden states from the forward and backward directions are concatenated before classification.

### Architecture

`Token IDs → Embedding → BiLSTM → Concatenated Hidden States → Linear → ReLU → Linear → Sigmoid`


In [16]:
import torch.nn as nn

EMBED_DIM = 20
RNN_HIDDEN_SIZE = 64
FC_HIDDEN_SIZE = 64

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=rnn_hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(rnn_hidden_size * 2, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text, lengths):
        embedded = self.embedding(text)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        hidden = torch.cat((forward_hidden, backward_hidden), dim=1)

        output = self.fc1(hidden)
        output = self.relu(output)
        output = self.fc2(output)
        output = self.sigmoid(output)

        return output.squeeze(1)

model = BiLSTM(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    rnn_hidden_size=RNN_HIDDEN_SIZE,
    fc_hidden_size=FC_HIDDEN_SIZE
).to(DEVICE)

print(model)


BiLSTM(
  (embedding): Embedding(75957, 20, padding_idx=0)
  (lstm): LSTM(20, 64, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 11. Loss Function and Optimizer

Binary cross-entropy is used because the task has two classes: **negative (0)** and **positive (1)**.


In [ ]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002
)


## 12. Training and Evaluation Functions


In [ ]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for text, lengths, labels in loader:
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        predictions = model(text, lengths)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_correct += ((predictions >= 0.5).float() == labels).sum().item()
        total_samples += len(labels)

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for text, lengths, labels in loader:
            text = text.to(DEVICE)
            labels = labels.to(DEVICE)

            predictions = model(text, lengths)
            loss = criterion(predictions, labels)

            total_loss += loss.item() * len(labels)
            total_correct += ((predictions >= 0.5).float() == labels).sum().item()
            total_samples += len(labels)

    return total_loss / total_samples, total_correct / total_samples


## 13. Train the BiLSTM

The final experiment uses **10 training epochs**. Training and validation loss/accuracy are recorded for every epoch.


In [23]:
NUM_EPOCHS = 10

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(valid_loss)
    history["val_accuracy"].append(valid_acc)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {valid_loss:.4f} | "
        f"Val Acc: {valid_acc:.4f}"
    )


Epoch 1/10 | Train Loss: 0.1551 | Train Acc: 0.9416 | Val Loss: 0.3850 | Val Acc: 0.8380
Epoch 2/10 | Train Loss: 0.1030 | Train Acc: 0.9653 | Val Loss: 0.4619 | Val Acc: 0.8452
Epoch 3/10 | Train Loss: 0.0737 | Train Acc: 0.9752 | Val Loss: 0.5152 | Val Acc: 0.8450
Epoch 4/10 | Train Loss: 0.0490 | Train Acc: 0.9849 | Val Loss: 0.5570 | Val Acc: 0.8444
Epoch 5/10 | Train Loss: 0.0272 | Train Acc: 0.9917 | Val Loss: 0.6482 | Val Acc: 0.8414
Epoch 6/10 | Train Loss: 0.0171 | Train Acc: 0.9958 | Val Loss: 0.8687 | Val Acc: 0.8440
Epoch 7/10 | Train Loss: 0.0193 | Train Acc: 0.9933 | Val Loss: 0.8639 | Val Acc: 0.8460
Epoch 8/10 | Train Loss: 0.0448 | Train Acc: 0.9842 | Val Loss: 0.7029 | Val Acc: 0.8442
Epoch 9/10 | Train Loss: 0.0132 | Train Acc: 0.9962 | Val Loss: 0.9029 | Val Acc: 0.8472
Epoch 10/10 | Train Loss: 0.0654 | Train Acc: 0.9764 | Val Loss: 0.6866 | Val Acc: 0.8452


### Training observations

The recorded run shows the training accuracy increasing substantially while validation accuracy levels off around the mid-80% range. This indicates that the model learns the training set very well, while additional epochs provide limited validation improvement.


## 14. Training Curves

The following cell visualizes the recorded training and validation metrics when the notebook is executed from start to finish.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Validation Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["train_accuracy"], label="Train Accuracy")
axes[1].plot(history["val_accuracy"], label="Validation Accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 15. Final Test Evaluation

The held-out test set contains 25,000 reviews and is evaluated only after training is complete.


In [24]:
test_loss, test_acc = evaluate(model, test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


Test Loss: 0.7689
Test Accuracy: 0.8291


### Final result

**Test accuracy: 82.91%** on the 25,000-review IMDB test set for the recorded 10-epoch run.


## 16. Save the Trained Model

The checkpoint stores the model weights and the configuration required to reconstruct the same architecture during inference.


In [26]:
import os

os.makedirs("models", exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab_size": len(vocab),
    "embed_dim": EMBED_DIM,
    "rnn_hidden_size": RNN_HIDDEN_SIZE,
    "fc_hidden_size": FC_HIDDEN_SIZE,
    "pad_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
    "max_length": MAX_LENGTH,
    "test_accuracy": test_acc
}

torch.save(checkpoint, "models/sentiment_bilstm.pth")
print("Model saved to models/sentiment_bilstm.pth")


Model saved.


## 17. Save the Vocabulary

The vocabulary must be kept with the model because inference uses the same token-to-index mapping that was created during training.


In [27]:
import json

with open("models/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f)

print("Vocabulary saved to models/vocab.json")


Vocabulary saved.


## 18. Project Artifacts

After running the notebook, the `models/` directory contains:

```text
models/
├── sentiment_bilstm.pth
└── vocab.json
```

These files are used by the separate inference/demo script on the Ubuntu laptop.

> **Colab note:** The following optional cells download the artifacts to the local computer when this notebook is opened in Google Colab. They are not required for model training.


In [ ]:
# Optional: download the trained model from Google Colab
from google.colab import files
files.download("models/sentiment_bilstm.pth")


In [ ]:
# Optional: download the vocabulary from Google Colab
files.download("models/vocab.json")


## 19. Conclusion

This project implements an end-to-end sentiment classification pipeline using a Bidirectional LSTM. The model was trained on 20,000 IMDB reviews, validated on 5,000 reviews, and evaluated on the 25,000-review test set. The recorded 10-epoch experiment achieved **82.91% test accuracy**.

The trained checkpoint and vocabulary can be transferred to another machine for lightweight inference without retraining the model.

### Possible improvements

- Use pretrained word embeddings.
- Tune the vocabulary size and sequence length.
- Add dropout and regularization.
- Use early stopping based on validation performance.
- Compare the BiLSTM with GRU, CNN, or transformer-based models.
